In [4]:
import pygame
import math
import os
pygame.init()

width, height = 750, 500
center_x = width // 2
center_y = height // 2

VR0 = -3
R0 = 200

TAIL_LENGTH = 400      # number of stored points
TAIL_FADE_DIST = TAIL_LENGTH + 100  # fade scale (distance ahead)

G = 1
c = 1
M0 = 40000
EH_R = 10
L = 6
dt = 0.1

white = (255,255,255)
black = (0,0,0)
orange = (255,165,0)

WIN = pygame.display.set_mode((width,height))
pygame.display.set_caption("planet in GR - fading tail")


def M_of_r(r):
    return M0 * (r**(1/2))


def rphi_to_xy(r, phi):
    x = center_x + r*math.cos(phi)
    y = center_y + r*math.sin(phi)
    return x, y


def derivatives(r, phi, vr, omega):
    M = M_of_r(r)
    Fr = -(G*M/r**2)*(1 + (3*L**2/(c**2*r**2)))

    drdt = vr
    dphidt = omega
    dvrdt = r*omega**2 + Fr
    domegadt = -2*vr*omega/r

    return drdt, dphidt, dvrdt, domegadt


def RK4(r, phi, vr, omega):

    k1 = derivatives(r, phi, vr, omega)

    k2 = derivatives(
        r + k1[0]*dt/2,
        phi + k1[1]*dt/2,
        vr + k1[2]*dt/2,
        omega + k1[3]*dt/2
    )

    k3 = derivatives(
        r + k2[0]*dt/2,
        phi + k2[1]*dt/2,
        vr + k2[2]*dt/2,
        omega + k2[3]*dt/2
    )

    k4 = derivatives(
        r + k3[0]*dt,
        phi + k3[1]*dt,
        vr + k3[2]*dt,
        omega + k3[3]*dt
    )

    dr = dt*(k1[0] + 2*k2[0] + 2*k3[0] + k4[0])/6
    dphi = dt*(k1[1] + 2*k2[1] + 2*k3[1] + k4[1])/6
    dvr = dt*(k1[2] + 2*k2[2] + 2*k3[2] + k4[2])/6
    domega = dt*(k1[3] + 2*k2[3] + 2*k3[3] + k4[3])/6

    return dr, dphi, dvr, domega


class Planet:
    def __init__(self, r, phi, vr, omega):
        self.r = r
        self.phi = phi
        self.vr = vr
        self.omega = omega
        self.path = []

    def update(self):
        if self.r > EH_R:
            dr, dphi, dvr, domega = RK4(self.r, self.phi, self.vr, self.omega)
            self.r += dr
            self.phi += dphi
            self.vr += dvr
            self.omega += domega

        x, y = rphi_to_xy(self.r, self.phi)
        self.path.append((x,y))

        if len(self.path) > TAIL_LENGTH:
            self.path.pop(0)

    def draw(self, win):
        x, y = rphi_to_xy(self.r, self.phi)

        # draw fading tail by age (never re-brightens on close passes)
        n = len(self.path)
        tail_surface = pygame.Surface((width, height), pygame.SRCALPHA)
        for i in range(1, n):
            p1 = self.path[i-1]
            p2 = self.path[i]
            age_t = i / max(1, n - 1)
            alpha = int(180 * (age_t**1.8))
            color = (255, 255, 255, alpha)
            pygame.draw.line(tail_surface, color, p1, p2, 2)

        win.blit(tail_surface, (0, 0))

        pygame.draw.circle(win, orange, (int(x),int(y)), 5)


class Mass:
    def __init__(self, R):
        self.x = center_x
        self.y = center_y
        self.R = R

    def draw(self, win):
        pygame.draw.circle(win, white, (int(self.x),int(self.y)), int(self.R))


def main():
    os.makedirs("frames", exist_ok=True)
    frame_number = 0
    MAX_FRAMES = 600   # for example
    frames = []
    run = True
    clock = pygame.time.Clock()

    black_hole = Mass(EH_R)

    r0 = R0
    phi0 = 0
    vr0 = VR0

    M_init = M_of_r(r0)
    omega0 = math.sqrt(G * M_init / r0**3)


    planet = Planet(r0, phi0, vr0, omega0)

    while run:
        clock.tick(60)
        WIN.fill(black)

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                run = False

        black_hole.draw(WIN)
        planet.update()
        planet.draw(WIN)
        if frame_number < MAX_FRAMES:
            pygame.image.save(WIN, f"frames/frame_{frame_number:04d}.png")
            frame_number += 1
        pygame.display.update()

    pygame.quit()
main()


In [21]:
import pygame
import math
import os
pygame.init()

width, height = 1200, 800
center_x = width/2
center_y = height/2

G = 1
c = 1
M = 40000   #100
# EH_R = 2*G*M/c**2
EH_R = 10     #0.6
L = 6         #1
    
dt = 0.1 
PIXELS_PER_UNIT = 1     #15
# startx, starty = 200, 300
# startx = center_x + 12 * PIXELS_PER_UNIT
# starty = center_y

TAIL_LENGTH = 500      # number of stored points
TAIL_FADE_DIST = TAIL_LENGTH + 500   # fade scale (distance ahead)

white = (255, 255, 255)
black = (0, 0, 0)
orange = (255, 165, 0)

WIN =pygame.display.set_mode((width, height))
pygame.display.set_caption("planet in GR - RK4")



def xy_to_rphi(x, y):
    # dx = (x - center_x) / PIXELS_PER_UNIT
    # dy = (y - center_y) / PIXELS_PER_UNIT
    dx = (x - center_x) 
    dy = (y - center_y)
    r = math.sqrt(dx*dx + dy*dy)
    phi = math.atan2(dy, dx)
    return r, phi

# def rphi_to_xy(r, phi):
#     x = center_x + r * math.cos(phi) * PIXELS_PER_UNIT
#     y = center_y + r * math.sin(phi) * PIXELS_PER_UNIT
#     return x, y




def calculations(x, y, vx, vy):
    r, phi = xy_to_rphi(x , y)
    #Fx = -(G*M/r**2)*(1 + (3*L**2 / (c**2*r**2)))* math.cos(phi) 
    #Fy = -(G*M/r**2)*(1 + (3*L**2 / (c**2*r**2)))* math.sin(phi)   

    Fx = -(G*M/r**2)* math.cos(phi) 
    Fy = -(G*M/r**2)* math.sin(phi)

    # vx =+ Fx*dt
    # vy =+ Fy*dt


    return vx, vy, Fx, Fy




def RK4(x, y, vx, vy):
    
    k1 = calculations(x, y, vx, vy)
    x1 = x + k1[0]*dt/2
    y1 = y + k1[1]*dt/2
    vx1 = vx + k1[2]*dt/2
    vy1 = vy + k1[3]*dt/2

    k2 = calculations(x1, y1, vx1, vy1)
    x2 = x + k2[0]*dt/2
    y2 = y + k2[1]*dt/2
    vx2 = vx + k2[2]*dt/2
    vy2 = vy + k2[3]*dt/2


    k3 = calculations(x2, y2, vx2, vy2)
    x3 = x + k3[0]*dt
    y3 = y + k3[1]*dt
    vx3 = vx + k3[2]*dt
    vy3 = vy + k3[3]*dt


    k4 = calculations(x3, y3, vx3, vy3)

    d_x = dt*(k1[0] + 2* k2[0] + 2* k3[0] + k4[0])/6
    d_y = dt*(k1[1] + 2* k2[1] + 2* k3[1] + k4[1])/6
    dv_x = dt*(k1[2] + 2* k2[2] + 2* k3[2] + k4[2])/6
    dv_y = dt*(k1[3] + 2* k2[3] + 2* k3[3] + k4[3])/6
        

    return d_x, d_y, dv_x, dv_y





class Planet:
    def __init__(self, x, y, vx, vy):
        self.x = x 
        self.y = y 
        self.vx = vx
        self.vy = vy
        self.path = []

    def draw(self, win):
        """Draw the planet and its fading tail."""
        px = int(self.x)
        py = int(self.y)

        # draw fading tail
        if len(self.path) > 1:
            for i in range(1, len(self.path)):
                p1 = self.path[i-1]
                p2 = self.path[i]
                dx = px - p1[0]
                dy = py - p1[1]
                dist = math.hypot(dx, dy)
                alpha = max(0, 255 * (1 - dist/TAIL_FADE_DIST))
                color = (int(alpha), int(alpha), int(alpha))
                pygame.draw.line(win, color, p1, p2, 2)

        pygame.draw.circle(win, orange, (px, py), 5)

    def update_position(self):
        """Advance the planet one timestep with RK4."""
        x = self.x 
        y = self.y
        vx = self.vx
        vy = self.vy 
        r, phi = xy_to_rphi(x, y)

        d_x, d_y, dv_x, dv_y = RK4(x, y, vx, vy)
        if r > EH_R:
            self.x += d_x
            self.y += d_y
            self.vx += dv_x
            self.vy += dv_y

        self.path.append((self.x, self.y))
        if len(self.path) > TAIL_LENGTH:
            self.path.pop(0)

class mass:
    def __init__(self, R):
        self.x = center_x
        self.y = center_y
        self.R = R

    def draw_c(self, win):
        pygame.draw.circle(win, white, (self.x, self.y), self.R)




def main():
    
    os.makedirs("frames", exist_ok=True)
    frame_number = 0
    MAX_FRAMES = 900   # for example
    frames = []
    run = True
    
    clock = pygame.time.Clock()

    # black_hole = mass(EH_R * PIXELS_PER_UNIT)
    black_hole = mass(EH_R)

    planet1 = Planet(450, 250, 10.5, 0)
    
    

    while run:
        clock.tick(60)

        WIN.fill(black)
        # pygame.display.update()

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                run = False

        black_hole.draw_c(WIN)

        planet1.draw(WIN)
        planet1.update_position()
        if frame_number < MAX_FRAMES:
            pygame.image.save(WIN, f"frames/frame_{frame_number:04d}.png")
            frame_number += 1
        pygame.display.update()

    pygame.quit()

main()

In [22]:
import numpy as np
import plotly.graph_objects as go



def calculate_turning_path(num_layers, total_h=10, n_start=1.0, n_end=0.1):
    """Calculates a ray that bends from an initial angle to a horizontal path."""
    theta_init_rad = np.radians(25) # Starts at 25 degrees from the normal
    n_values = np.linspace(n_start, n_end, num_layers + 1)
    dy = total_h / num_layers
    
    # Start at the top edge (Y=5)
    x = [0]
    y = [total_h / 2]
    
    curr_x, curr_y = 0, total_h / 2
    curr_theta = theta_init_rad
    is_parallel = False

    for i in range(num_layers):
        if not is_parallel:
            # Snell's Law: n1 * sin(t1) = n2 * sin(t2)
            # Rearranged: sin(t2) = (n_in / n_out) * sin(t1)
            sin_out = (n_values[i] * np.sin(curr_theta)) / n_values[i+1]
            
            if sin_out >= 1.0:
                # Critical angle reached! Ray turns parallel.
                is_parallel = True
                # Move to the boundary where it turned
                curr_x += dy * np.tan(np.radians(85)) # Near-parallel step
                curr_y -= dy
            else:
                curr_theta = np.arcsin(sin_out)
                curr_x += dy * np.tan(curr_theta)
                curr_y -= dy
            
            x.append(curr_x)
            y.append(curr_y)
        else:
            # Ray is now parallel; keep the same Y and move to the right edge
            pass 

    # Final horizontal extension to the edge of the graph
    x.append(30)
    y.append(y[-1])
        
    return x, y

# --- 1. Prepare Plotly Steps ---
resolutions = [1, 2, 4, 8, 16, 32, 64, 128]
fig = go.Figure()

for i, res in enumerate(resolutions):
    xr, yr = calculate_turning_path(res)
    fig.add_trace(go.Scatter(
        x=xr, y=yr, mode='lines',
        line=dict(color='cyan', width=4, shape='spline' if res > 20 else 'linear'),
        visible=(i == 0)
    ))

# --- 2. Generate Gradient Blocks ---
all_shape_sets = []
for res in resolutions:
    dy = 10 / res
    shapes = []
    for j in range(res):
        shapes.append(dict(
            type="rect", x0=-2, x1=30, y0=5-(j+1)*dy, y1=5-j*dy,
            fillcolor="purple", opacity=min(0.05 + (j/res)*0.4, 0.5),
            line=dict(color="rgba(255,255,255,0.1)", width=0.5 if res < 32 else 0),
            layer="below"
        ))
    all_shape_sets.append(shapes)

# --- 3. Slider Setup ---
steps = []
for i in range(len(resolutions)):
    steps.append(dict(
        method="update", label=str(resolutions[i]),
        args=[{"visible": [j == i for j in range(len(resolutions))]},
              {"shapes": all_shape_sets[i]}]
    ))

# --- 4. Layout ---
fig.update_layout(
    title="Plasma Refraction: The Transition to Parallel (Total Reflection)",
    template="plotly_dark",
    xaxis=dict(visible=False, range=[-1, 30]),
    yaxis=dict(visible=False, range=[-5.5, 5.5]),
    sliders=[dict(active=0, steps=steps, currentvalue={"prefix": "Resolution: "}, pad={"t": 50})],
    width=800,
    height=420,
    margin=dict(l=20, r=20, t=60, b=20)
)

fig.show()



ModuleNotFoundError: No module named 'plotly'